# CardioVision 3D — Data Preprocessing

## 1. Data Loading and Preparation

This notebook focuses on preparing the raw dataset for machine learning while preserving the original data.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Load original dataset
file_path = Path("../dataset/raw/extention of Z-Alizadeh sani dataset.xlsx")

df = pd.read_excel(file_path)

print("Dataset Shape:", df.shape)
display(df.head())

In [ ]:
# Target columns
target_columns = ["Cath", "LAD", "LCX", "RCA"]

# Separate input features and targets
X = df.drop(columns=target_columns)
y = df[target_columns]

print("Input Features Shape:", X.shape)
print("Target Variables Shape:", y.shape)

print("\nTarget Columns:", y.columns.tolist())
print("Target columns inside X:", set(target_columns).intersection(X.columns))

In [ ]:
for column in target_columns:
    print(f"\n{column} Distribution:")
    print(y[column].value_counts())

    print("\nPercentage:")
    print((y[column].value_counts(normalize=True) * 100).round(2))

In [ ]:
target_summary = []

for column in target_columns:
    counts = y[column].value_counts()
    percentages = y[column].value_counts(normalize=True) * 100

    for category in counts.index:
        target_summary.append({
            "Target": column,
            "Class": category,
            "Count": counts[category],
            "Percentage": round(percentages[category], 2)
        })

target_summary_df = pd.DataFrame(target_summary)
display(target_summary_df)

In [ ]:
feature_summary = pd.DataFrame({
    "Feature Type": ["Numerical", "Categorical"],
    "Count": [
        X.select_dtypes(include=np.number).shape[1],
        X.select_dtypes(exclude=np.number).shape[1]
    ]
})

display(feature_summary)

In [ ]:
missing_values = X.isnull().sum()

print("Total Missing Values:", missing_values.sum())
print("\nColumns with missing values:")
display(missing_values[missing_values > 0])

In [ ]:
constant_features = [
    column for column in X.columns
    if X[column].nunique() == 1
]

print("Constant Features:", constant_features)

In [ ]:
constant_features = [
    column for column in X.columns
    if X[column].nunique() == 1
]

print("Constant Features:", constant_features)
print("Number of Constant Features:", len(constant_features))

In [ ]:
for column in constant_features:
    print(f"\n{column}:")
    print(X[column].value_counts())

## 2. Initial Data Quality Checks

- Missing Values: 0
- Constant Features: 1 (`Exertional CP`)
- Numerical Features: 34
- Categorical Features: 21
- Target Leakage: All four target columns successfully separated from input features.

**Preprocessing Decision:** The constant feature `Exertional CP` will be excluded from model input features because it contains no variation.

In [ ]:
# Remove constant features from model inputs
X_clean = X.drop(columns=constant_features)

print("Original Input Shape:", X.shape)
print("Clean Input Shape:", X_clean.shape)
print("Target Shape:", y.shape)

In [ ]:
categorical_columns = X_clean.select_dtypes(
    exclude=np.number
).columns.tolist()

for column in categorical_columns:
    print(f"\n{column}:")
    print(X_clean[column].value_counts())

In [ ]:
numerical_columns = X_clean.select_dtypes(
    include=np.number
).columns.tolist()

display(X_clean[numerical_columns].describe().T)

In [ ]:
for column in categorical_columns:
    values = X_clean[column].dropna().astype(str)

    inconsistent = values[
        values.str.strip() != values
    ]

    if not inconsistent.empty:
        print(f"{column}:")
        print(inconsistent.unique())

In [ ]:
numerical_summary = X_clean[numerical_columns].describe().T

numerical_summary.to_csv(
    "../reports/numerical_feature_summary.csv"
)

print("Numerical summary saved successfully!")

In [ ]:
categorical_summary = pd.DataFrame({
    "Column": categorical_columns,
    "Unique Values": [
        X_clean[col].nunique() for col in categorical_columns
    ],
    "Categories": [
        X_clean[col].unique().tolist() for col in categorical_columns
    ]
})

display(categorical_summary)

In [ ]:
print(X_clean["Sex"].value_counts())
print("\nUnique values:", X_clean["Sex"].unique())

In [ ]:
categorical_summary.to_csv(
    "../reports/categorical_feature_summary.csv",
    index=False
)

print("Categorical summary saved successfully!")

In [ ]:
binary_categorical = [
    col for col in categorical_columns
    if X_clean[col].nunique() == 2
]

print("Binary Categorical Features:", binary_categorical)
print("Count:", len(binary_categorical))

In [ ]:
multi_categorical = [
    col for col in categorical_columns
    if X_clean[col].nunique() > 2
]

print("Multi-category Features:", multi_categorical)
print("Count:", len(multi_categorical))

In [ ]:
encoding_plan = pd.DataFrame({
    "Feature": binary_categorical + multi_categorical,
    "Category Type": (
        ["Binary"] * len(binary_categorical) +
        ["Multi-category"] * len(multi_categorical)
    )
})

display(encoding_plan)

## 3. Categorical Encoding Strategy

- Binary categorical features will be encoded using appropriate binary encoding.
- Multi-category features (`BBB`, `VHD`) will use One-Hot Encoding.
- The `Sex` feature contains an unverified category (`Fmale`).
- No category correction will be performed without reliable source evidence.
- Encoding will be implemented inside a machine learning preprocessing pipeline to prevent data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

print("Scikit-learn preprocessing imports successful!")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

print("Scikit-learn preprocessing imports successful!")

In [ ]:
# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both transformers
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_columns),
        ("cat", categorical_transformer, categorical_columns)
    ]
)

print("Preprocessing pipeline created successfully!")

In [ ]:
print(preprocessor)

In [ ]:
numeric_binary_columns = [
    "DM", "HTN", "Current Smoker", "EX-Smoker", "FH",
    "Edema", "Typical Chest Pain", "Q Wave",
    "St Elevation", "St Depression", "Tinversion"
]

numeric_ordinal_columns = [
    "Function Class", "Region RWMA"
]

continuous_columns = [
    col for col in numerical_columns
    if col not in numeric_binary_columns + numeric_ordinal_columns
]

print("Continuous:", len(continuous_columns))
print("Binary:", len(numeric_binary_columns))
print("Ordinal:", len(numeric_ordinal_columns))

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", StandardScaler(), continuous_columns),
        ("binary", "passthrough", numeric_binary_columns),
        ("ordinal", "passthrough", numeric_ordinal_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore"),
         categorical_columns)
    ]
)

print("Improved preprocessing pipeline created!")

In [ ]:
print("Total input columns:", len(
    continuous_columns + numeric_binary_columns +
    numeric_ordinal_columns + categorical_columns
))
print("Expected input columns:", X_clean.shape[1])

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y["Cath"],
    test_size=0.20,
    random_state=42,
    stratify=y["Cath"]
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y["Cath"],
    test_size=0.20,
    random_state=42,
    stratify=y["Cath"]
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

In [ ]:
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Total records:", len(X_train) + len(X_test))

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

print("Training preprocessing completed!")
print("Processed Training Shape:", X_train_processed.shape)

In [ ]:
X_test_processed = preprocessor.transform(X_test)

print("Testing preprocessing completed!")
print("Processed Testing Shape:", X_test_processed.shape)

In [ ]:
import numpy as np

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

print("Missing values in training:",
      np.isnan(X_train_processed).sum())

print("Missing values in testing:",
      np.isnan(X_test_processed).sum())

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train model
lr_model.fit(X_train_processed, y_train)

print("Logistic Regression model trained successfully!")

In [ ]:
# Generate predictions on unseen test data
y_pred = lr_model.predict(X_test_processed)

print("Predictions generated successfully!")
print("First 10 Predictions:", y_pred[:10])
print("First 10 Actual Values:", y_test.iloc[:10].values)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_pred,
    zero_division=0
)

print(report)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["CAD", "Normal"]
)

display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["CAD", "Normal"]
)

display_matrix.plot(cmap="Blues")
plt.title("Logistic Regression - Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

evaluation_results = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred),
    "CAD Precision": precision_score(
        y_test, y_pred, pos_label="CAD"
    ),
    "CAD Recall": recall_score(
        y_test, y_pred, pos_label="CAD"
    ),
    "CAD F1 Score": f1_score(
        y_test, y_pred, pos_label="CAD"
    )
}])

display(evaluation_results)

In [ ]:
evaluation_results.to_csv(
    "../reports/model_evaluation.csv",
    index=False
)

print("Model evaluation saved successfully!")

In [ ]:
import joblib

joblib.dump(
    lr_model,
    "../ml/models/logistic_regression_cad.pkl"
)

print("Logistic Regression model saved successfully!")

In [ ]:
joblib.dump(
    preprocessor,
    "../ml/pipelines/cad_preprocessor.pkl"
)

print("CAD preprocessing pipeline saved successfully!")

In [ ]:
loaded_model = joblib.load(
    "../ml/models/logistic_regression_cad.pkl"
)

print("Saved model loaded:", type(loaded_model).__name__)

In [ ]:
loaded_predictions = loaded_model.predict(X_test_processed)

print(
    "Predictions Match:",
    np.array_equal(y_pred, loaded_predictions)
)

## 4. Baseline Model — Logistic Regression

### Model Configuration
- Algorithm: Logistic Regression
- Target: Cath (CAD Classification)
- Training Records: 242
- Testing Records: 61
- Processed Features: 77

### Evaluation Results
- Accuracy: 81.97%
- CAD Precision: 84.78%
- CAD Recall: 90.70%
- CAD F1-score: 87.64%

### Model Artifacts
- Model: `ml/models/logistic_regression_cad.pkl`
- Preprocessor: `ml/pipelines/cad_preprocessor.pkl`
- Evaluation: `reports/model_evaluation.csv`

### Verification
- Model reload: Successful
- Prediction consistency: Passed

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# Train model
rf_model.fit(X_train_processed, y_train)

print("Random Forest model trained successfully!")

In [ ]:
rf_predictions = rf_model.predict(X_test_processed)

print("Random Forest predictions generated!")
print("First 10 Predictions:", rf_predictions[:10])
print("First 10 Actual Values:", y_test.iloc[:10].values)

In [ ]:
from sklearn.metrics import accuracy_score

rf_accuracy = accuracy_score(y_test, rf_predictions)

print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%")

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    rf_predictions,
    zero_division=0
))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

rf_cm = confusion_matrix(
    y_test,
    rf_predictions,
    labels=["CAD", "Normal"]
)

rf_display = ConfusionMatrixDisplay(
    confusion_matrix=rf_cm,
    display_labels=["CAD", "Normal"]
)

rf_display.plot(cmap="Blues")
plt.title("Random Forest - Confusion Matrix")
plt.show()

In [ ]:
rf_evaluation = pd.DataFrame([{
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, rf_predictions),
    "CAD Precision": precision_score(
        y_test, rf_predictions, pos_label="CAD"
    ),
    "CAD Recall": recall_score(
        y_test, rf_predictions, pos_label="CAD"
    ),
    "CAD F1 Score": f1_score(
        y_test, rf_predictions, pos_label="CAD"
    )
}])

model_comparison = pd.concat(
    [evaluation_results, rf_evaluation],
    ignore_index=True
)

display(model_comparison)

In [ ]:
model_comparison.to_csv(
    "../reports/model_comparison.csv",
    index=False
)

print("Model comparison report saved successfully!")

In [ ]:
joblib.dump(
    rf_model,
    "../ml/models/random_forest_cad.pkl"
)

print("Random Forest model saved successfully!")

In [ ]:
loaded_rf_model = joblib.load(
    "../ml/models/random_forest_cad.pkl"
)

print("Loaded Model:", type(loaded_rf_model).__name__)

In [ ]:
loaded_rf_predictions = loaded_rf_model.predict(
    X_test_processed
)

print(
    "Predictions Match:",
    np.array_equal(rf_predictions, loaded_rf_predictions)
)

## 5. Random Forest — Baseline Model

### Model Configuration
- Algorithm: Random Forest Classifier
- Target: Cath (CAD Classification)
- Number of Trees: 200
- Training Records: 242
- Testing Records: 61

### Evaluation Results
- Accuracy: 83.61%
- CAD Precision: 86.67%
- CAD Recall: 90.70%
- CAD F1-score: 88.64%

### Model Artifact
- Model: `ml/models/random_forest_cad.pkl`

### Verification
- Model reload: Successful
- Prediction consistency: Passed

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Cross-validation imports successful!")

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Number of folds:", cv.get_n_splits())

In [ ]:
rf_cv_scores = cross_val_score(
    rf_model,
    X_train_processed,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("Cross-Validation Scores:", rf_cv_scores)

In [ ]:
print(
    f"Mean CV Accuracy: {rf_cv_scores.mean() * 100:.2f}%"
)

print(
    f"Standard Deviation: {rf_cv_scores.std() * 100:.2f}%"
)

In [ ]:
cv_results = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": rf_cv_scores
})

cv_results.to_csv(
    "../reports/random_forest_cv_results.csv",
    index=False
)

print("Cross-validation results saved successfully!")

In [ ]:
cv_summary = pd.DataFrame([{
    "Model": "Random Forest",
    "CV Folds": 5,
    "Mean CV Accuracy": rf_cv_scores.mean(),
    "CV Standard Deviation": rf_cv_scores.std()
}])

cv_summary.to_csv(
    "../reports/random_forest_cv_summary.csv",
    index=False
)

display(cv_summary)
print("CV summary saved successfully!")

In [ ]:
import matplotlib.pyplot as plt

models = model_comparison["Model"]
accuracies = model_comparison["Accuracy"] * 100

plt.figure(figsize=(8, 5))
plt.bar(models, accuracies)

plt.title("CAD Prediction - Model Accuracy Comparison")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)

for i, value in enumerate(accuracies):
    plt.text(i, value + 1, f"{value:.2f}%", ha="center")

plt.savefig(
    "../reports/model_accuracy_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": preprocessor.get_feature_names_out(),
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

display(feature_importance.head(15))

In [ ]:
feature_importance.to_csv(
    "../reports/random_forest_feature_importance.csv",
    index=False
)

print("Feature importance report saved successfully!")

In [ ]:
top_features = feature_importance.head(15).sort_values(
    by="Importance",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.title("Top 15 Features - Random Forest")
plt.xlabel("Feature Importance")
plt.ylabel("Features")

plt.tight_layout()

plt.savefig(
    "../reports/random_forest_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import shap

print("SHAP Version:", shap.__version__)
print("SHAP imported successfully!")

In [ ]:
import shap

print("SHAP Version:", shap.__version__)
print("SHAP imported successfully!")

In [ ]:
required_variables = [
    "shap",
    "rf_model",
    "X_train_processed",
    "X_test_processed",
    "y_train",
    "y_test",
    "preprocessor"
]

for variable in required_variables:
    print(f"{variable}: {variable in globals()}")

In [ ]:
import shap
import joblib
import pandas as pd
import numpy as np

print("SHAP:", shap.__version__)
print("Required libraries imported successfully!")

In [ ]:
rf_model = joblib.load(
    "../ml/models/random_forest_cad.pkl"
)

print("Model loaded:", type(rf_model).__name__)

In [ ]:
preprocessor = joblib.load(
    "../ml/pipelines/cad_preprocessor.pkl"
)

print("Preprocessor loaded:", type(preprocessor).__name__)

In [ ]:
from pathlib import Path

file_path = Path(
    "../dataset/raw/extention of Z-Alizadeh sani dataset.xlsx"
)

df = pd.read_excel(file_path)

print("Dataset Shape:", df.shape)

In [ ]:
target_columns = ["Cath", "LAD", "LCX", "RCA"]

X = df.drop(columns=target_columns)
y = df[target_columns]

# Remove the previously identified constant feature
X_clean = X.drop(columns=["Exertional CP"])

print("Clean Input Shape:", X_clean.shape)
print("Target Shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y["Cath"],
    test_size=0.20,
    random_state=42,
    stratify=y["Cath"]
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


In [ ]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed Train Shape:", X_train_processed.shape)
print("Processed Test Shape:", X_test_processed.shape)

In [ ]:
print("Model Classes:", rf_model.classes_)

In [ ]:
explainer = shap.TreeExplainer(rf_model)

print("SHAP Explainer Created Successfully!")

In [ ]:
shap_values = explainer.shap_values(X_test_processed[:10])

print("SHAP values calculated successfully!")
print("Output type:", type(shap_values))

if isinstance(shap_values, list):
    print("Number of class outputs:", len(shap_values))
    print("CAD SHAP shape:", shap_values[0].shape)
else:
    print("SHAP shape:", np.array(shap_values).shape)

In [ ]:
cad_shap_values = shap_values[:, :, 0]

feature_names = preprocessor.get_feature_names_out()

print("CAD SHAP Shape:", cad_shap_values.shape)
print("Number of Feature Names:", len(feature_names))

In [ ]:
import matplotlib.pyplot as plt

shap.summary_plot(
    cad_shap_values,
    X_test_processed[:10],
    feature_names=feature_names,
    show=False
)

plt.tight_layout()
plt.savefig(
    "../reports/shap_summary_cad.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
shap_values_all = explainer.shap_values(X_test_processed)

cad_shap_values_all = shap_values_all[:, :, 0]

print("All Test SHAP Shape:", cad_shap_values_all.shape)

In [ ]:
plt.figure()

shap.summary_plot(
    cad_shap_values_all,
    X_test_processed,
    feature_names=feature_names,
    show=False
)

plt.tight_layout()

plt.savefig(
    "../reports/shap_summary_cad_full.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Absolute_SHAP": np.abs(cad_shap_values_all).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    by="Mean_Absolute_SHAP",
    ascending=False
)

display(shap_importance.head(15))

shap_importance.to_csv(
    "../reports/shap_feature_importance.csv",
    index=False
)

In [ ]:
top_features = shap_importance.head(15).sort_values(
    by="Mean_Absolute_SHAP",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Mean_Absolute_SHAP"]
)

plt.xlabel("Mean Absolute SHAP Value")
plt.ylabel("Features")
plt.title("Top 15 Features - SHAP Importance (CAD Prediction)")

plt.tight_layout()

plt.savefig(
    "../reports/shap_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 6. Model Explainability Using SHAP

### Objective
SHAP (SHapley Additive exPlanations) was used to understand how individual input features influence the Random Forest model's CAD predictions.

### Methodology
- Used SHAP TreeExplainer for the trained Random Forest classifier.
- Generated SHAP values for all 61 test records.
- Analysed the CAD output class.
- Calculated mean absolute SHAP values to rank feature influence.

### Key Findings
1. Typical Chest Pain showed the highest mean absolute SHAP value (0.101079).
2. Age was the second most influential feature (0.049445).
3. Atypical_N, Atypical_Y, and Region RWMA were also among the top features.
4. SHAP summary plots were generated to examine feature impact and direction.

### Generated Reports
- shap_summary_cad_full.png
- shap_feature_importance.png
- shap_feature_importance.csv

### Limitations
- SHAP explanations describe model behaviour, not medical causation.
- Results are based on a small dataset of 303 records and a test set of 61 records.
- These predictions are exploratory and are not intended for clinical diagnosis.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

print("Leakage-safe Random Forest Pipeline created!")

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

pipeline_cv_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("Fold Accuracy Scores:", pipeline_cv_scores)
print("Mean CV Accuracy:", pipeline_cv_scores.mean())
print("Standard Deviation:", pipeline_cv_scores.std())

In [ ]:
cv_results = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": pipeline_cv_scores
})

cv_results.to_csv(
    "../reports/leakage_safe_cv_results.csv",
    index=False
)

cv_summary = pd.DataFrame({
    "Metric": ["Mean Accuracy", "Standard Deviation"],
    "Value": [
        pipeline_cv_scores.mean(),
        pipeline_cv_scores.std()
    ]
})

cv_summary.to_csv(
    "../reports/leakage_safe_cv_summary.csv",
    index=False
)

print("Cross-validation reports saved successfully!")

In [ ]:
rf_pipeline.fit(X_train, y_train)

y_test_pred = rf_pipeline.predict(X_test)

print("Final model training completed!")
print("Test predictions generated:", len(y_test_pred))

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_test_pred,
        labels=["CAD", "Normal"],
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(
    y_test,
    y_test_pred,
    labels=["CAD", "Normal"]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["CAD", "Normal"]
)

disp.plot(cmap="Blues", values_format="d")

plt.title("Random Forest - Final Test Confusion Matrix")
plt.tight_layout()

plt.savefig(
    "../reports/final_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Confusion Matrix:")
print(cm)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

final_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "CAD Precision",
        "CAD Recall",
        "CAD F1 Score",
        "Normal Precision",
        "Normal Recall",
        "Normal F1 Score"
    ],
    "Value": [
        accuracy_score(y_test, y_test_pred),
        precision_score(y_test, y_test_pred, pos_label="CAD"),
        recall_score(y_test, y_test_pred, pos_label="CAD"),
        f1_score(y_test, y_test_pred, pos_label="CAD"),
        precision_score(y_test, y_test_pred, pos_label="Normal"),
        recall_score(y_test, y_test_pred, pos_label="Normal"),
        f1_score(y_test, y_test_pred, pos_label="Normal")
    ]
})

display(final_metrics)

final_metrics.to_csv(
    "../reports/final_model_evaluation.csv",
    index=False
)

print("Final evaluation metrics saved successfully!")

In [ ]:
import joblib
from pathlib import Path

model_path = Path("../ml/models/final_cad_pipeline.pkl")

joblib.dump(rf_pipeline, model_path)

print("Final pipeline saved successfully!")
print("Saved at:", model_path.resolve())

In [ ]:
loaded_pipeline = joblib.load(
    "../ml/models/final_cad_pipeline.pkl"
)

reloaded_predictions = loaded_pipeline.predict(X_test)

print("Pipeline loaded:", type(loaded_pipeline))
print(
    "Prediction consistency:",
    np.array_equal(y_test_pred, reloaded_predictions)
)

In [ ]:
model_features = loaded_pipeline.feature_names_in_

print("Total Input Features:", len(model_features))
print("\nExpected Input Features:")

for index, feature in enumerate(model_features, start=1):
    print(f"{index}. {feature}")

In [ ]:
input_schema = pd.DataFrame({
    "position": range(1, len(model_features) + 1),
    "feature_name": model_features
})

display(input_schema)

input_schema.to_csv(
    "../ml/input_schema.csv",
    index=False
)

print("Input schema saved successfully!")

In [ ]:
import json

sample_input = (
    df.drop(
        columns=["Cath", "LAD", "LCX", "RCA", "Exertional CP"]
    )
    .iloc[0]
    .to_dict()
)

print(json.dumps(sample_input, indent=2, default=str))

In [ ]:
import pandas as pd
import json
from pathlib import Path

# Load original dataset
file_path = Path("../dataset/raw/extention of Z-Alizadeh sani dataset.xlsx")
df = pd.read_excel(file_path)

# Extract one original record without target leakage
sample_input = (
    df.drop(
        columns=["Cath", "LAD", "LCX", "RCA", "Exertional CP"]
    )
    .iloc[0]
    .to_dict()
)

print("Dataset shape:", df.shape)
print("Number of input features:", len(sample_input))
print("\nSample input:")
print(json.dumps(sample_input, indent=2, default=str))

In [ ]:
from pathlib import Path
import json

output_path = Path("../frontend/public/sample-patient.json")

output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(sample_input, file, indent=2)

print("Sample patient JSON saved successfully!")
print("File:", output_path.resolve())
print("Total features:", len(sample_input))